# Richmond POI and Infrastructure Review

This notebook is the evidence companion to `RICHMOND_POI_INFRA_REVIEW.md`.
It is meant for inspecting Richmond-first OSM infrastructure extracts and Overture POI extracts before we lock D4 source choices.

In [1]:
from pathlib import Path
import json

import duckdb
import pandas as pd

ROOT = Path.cwd()
if ROOT.name != 'industry':
    ROOT = ROOT / 'metro-deep-dive' / 'metro-area-explorer' / 'industry'

OUTPUT_DIR = ROOT / 'outputs' / 'richmond_va'
OUTPUT_DIR

PosixPath('/Users/danberle/Documents/projects/patterns_in_place/metro-deep-dive/metro-area-explorer/industry/outputs/richmond_va')

In [2]:
manifest_path = OUTPUT_DIR / 'spatial_manifest.json'
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {'layers': [], 'notes': ['Manifest not found yet.']}
pd.DataFrame(manifest.get('layers', []))

,source,layer_name,geometry_type,row_count,query_config,notes_on_sparse_or_missing_layers
0,osm,highways,unknown,0,"{'selectors': ['way[""highway""~""motorway|motorw...",<urlopen error [Errno 54] Connection reset by ...
1,osm,major_roads,unknown,0,"{'selectors': ['way[""highway""~""primary|primary...",<urlopen error [Errno 54] Connection reset by ...
2,osm,rail,unknown,0,"{'selectors': ['way[""railway""~""rail|light_rail...",<urlopen error [Errno 54] Connection reset by ...
3,osm,airports,unknown,0,"{'selectors': ['nwr[""aeroway""~""aerodrome|termi...",<urlopen error [Errno 54] Connection reset by ...
4,osm,ports,unknown,0,"{'selectors': ['nwr[""harbour""]({bbox});', 'nwr...",<urlopen error [Errno 54] Connection reset by ...
5,osm,warehouses_logistics,unknown,0,"{'selectors': ['nwr[""building""=""warehouse""]({b...",<urlopen error [Errno 54] Connection reset by ...
6,overture,overture_pois,mixed,76913,{'path': 's3://overturemaps-us-west-2/release/...,


## Load cached outputs

In [3]:
con = duckdb.connect()

def read_if_exists(filename):
    path = OUTPUT_DIR / filename
    if not path.exists():
        return pd.DataFrame()
    return con.execute('SELECT * FROM read_parquet(?)', [str(path)]).fetchdf()

osm_lines = read_if_exists('osm_infrastructure_lines.parquet')
osm_polygons = read_if_exists('osm_infrastructure_polygons.parquet')
osm_points = read_if_exists('osm_infrastructure_points.parquet')
overture_pois = read_if_exists('overture_pois.parquet')

{
    'osm_lines': len(osm_lines),
    'osm_polygons': len(osm_polygons),
    'osm_points': len(osm_points),
    'overture_pois': len(overture_pois),
}

{'osm_lines': 0, 'osm_polygons': 0, 'osm_points': 0, 'overture_pois': 76913}

## Layer summaries

In [4]:
def summarize(frame, name):
    if frame.empty:
        return pd.DataFrame([{'dataset': name, 'rows': 0, 'layer_group': None, 'category': None, 'subcategory': None}])
    summary = (
        frame.groupby(['layer_group', 'category', 'subcategory'], dropna=False)
        .size()
        .reset_index(name='rows')
    )
    summary.insert(0, 'dataset', name)
    return summary.sort_values('rows', ascending=False)

pd.concat([
    summarize(osm_lines, 'osm_lines'),
    summarize(osm_polygons, 'osm_polygons'),
    summarize(osm_points, 'osm_points'),
    summarize(overture_pois, 'overture_pois'),
], ignore_index=True)

,dataset,rows,layer_group,category,subcategory
0,osm_lines,0,None,None,None
1,osm_polygons,0,None,None,None
2,osm_points,0,None,None,None
3,overture_pois,4153,overture_pois,poi,home_service
4,overture_pois,4088,overture_pois,poi,restaurant
...,...,...,...,...,...
233,overture_pois,1,overture_pois,poi,rural_attraction
234,overture_pois,1,overture_pois,poi,public_fountain
235,overture_pois,1,overture_pois,poi,rodeo
236,overture_pois,1,overture_pois,poi,sculpture_statue


## Sample records

In [ ]:
osm_lines[['feature_name', 'layer_group', 'subcategory', 'attributes_json']].head(10) if not osm_lines.empty else pd.DataFrame()

In [ ]:
overture_pois[['feature_name', 'layer_group', 'category', 'subcategory', 'attributes_json']].head(10) if not overture_pois.empty else pd.DataFrame()

## Review prompts

- Are OSM highways / rail / airports / ports clean enough for D4 now?
- Are warehouse / logistics features useful enough to keep in the first wave?
- Do Overture hospitals and groceries have good enough coverage and labeling?
- Which Overture category strategy is better for our taxonomy work?
- Is bbox extraction acceptably clean, or do we need a clipping/filtering pass?